In [3]:
"""Module for implementing logistic regression using scikit-learn.
This module standardizes and transforms the suicide study dataset, 
and applies logistic regression to analyze trends in suicide data 
from both 2023 and 2013-2022 periods."""

import pandas as pd
import numpy as np

import sys
from pathlib import Path
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))

from src.config.config import DATA_DIR, RESULTS_DIR

In [ ]:
# ================================================================================
# Data reading
# ================================================================================
csv_file_path = RESULTS_DIR / "poLCA" / "Group_AG.csv"
lca_classes = pd.read_csv(
    csv_file_path, delimiter=",", low_memory=False, index_col=None
)

csv_file_path = DATA_DIR / "encoded" / "encoded_data.csv"
df_encoded = pd.read_csv(csv_file_path, delimiter=",", low_memory=False, index_col=None)

# merging
df_encoded = df_encoded.merge(lca_classes, on="ID", how="left")

groups = sorted(list(set(df_encoded["Group_AG"])))

In [ ]:
from sklearn.linear_model import LogisticRegression

results_data = []

for group in groups:
    # Select Group from Group_AG
    group_data = df_encoded[df_encoded["Group_AG"] == group]
    group_data["Predicted_Class_Group_AG"] = group_data[
        "Predicted_Class_Group_AG"
    ].astype("category")

    # Select columns
    X = group_data[["Predicted_Class_Group_AG"]]  # Zmienna niezależna
    y = group_data["Fatal"]  # Zmienna zależna

    X = pd.get_dummies(group_data["Predicted_Class_Group_AG"], drop_first=True)
    # Zakładamy, że 'Fatal' ma wartości 'True'/'False', przekształcamy na int
    y = y.astype(int)
    X = X.astype(int)

    # Używamy modelu regresji logistycznej
    logreg_model = LogisticRegression(
        max_iter=1000
    )  # Zwiększamy max_iter w razie potrzeby
    logreg_model.fit(X, y)

    log_likelihood = np.sum(np.log(logreg_model.predict_proba(X)[np.arange(len(y)), y]))
    # Przechowywanie wyników
    for col, coef in zip(X.columns, logreg_model.coef_[0]):
        results_data.append(
            {
                "Group": group,
                "Variable": col,
                "Coefficient": coef,
                "Intercept": logreg_model.intercept_[0],
                "Log-Likelihood": log_likelihood,
                "AIC": 2 * (X.shape[1] + 1) - 2 * log_likelihood,  # Aproksymacja AIC
                "BIC": np.log(len(y)) * (X.shape[1] + 1) - 2 * log_likelihood,
            }
        )

# Tworzenie DataFrame z wynikami
sklearn_results_df = pd.DataFrame(results_data)


In [ ]:
# Saveing
directory_path = RESULTS_DIR / "logreg"
file_name = "sklearn_logreg_results_ref.xlsx"
excel_file_path = directory_path / file_name

if not directory_path.exists():
    directory_path.mkdir(parents=True, exist_ok=True)

# Write the DataFrame to an Excel file
sklearn_results_df.to_excel(excel_file_path, index=False)